In [ ]:
#| default_exp fsrs

# FSRS

> The FSRS-6 memory model, ported from the fsrs-rs crate that Anki embeds

FSRS (Free Spaced Repetition Scheduler) models each card as a *memory state*, comprising a stability `S` (the interval, in days, at which recall probability falls to 90%) and a difficulty `D` in 1-10. A power-law forgetting curve — with a per-user shape parameter, `w20` — gives the probability of recall after any elapsed time. The next interval is wherever that curve crosses the user's desired retention, and each review updates `(S, D)` by the formulas below.

The heavy machinery in FSRS (the optimizer that fits the 21 parameters to a user's history) lives in the clients and reaches us as numbers in the deck preset, so fastanki doesn't need it.

This module is a direct port of the scheduling half of [fsrs-rs](https://github.com/open-spaced-repetition/fsrs-rs) (the crate compiled into Anki itself). The `scheduler` module wires it to the state machine. Collections in the wild carry parameters from three FSRS generations — 17 (FSRS-4.5, the `fsrsWeights` sync field), 19 (FSRS-5) or 21 (FSRS-6) numbers — and `fsrs_params` upgrades and range-clips them exactly as `FSRS::new` does, so a user who last optimized on an older Anki schedules identically here.

Note, the rust crate computes in 32-bit floats, Python in 64-bit. We squeeze each formula's *output* through float32 (`f32`), which keeps parity for everything that lands in the collection; Anki's own test suite acknowledges float variance across platforms and rounds before comparing, and the oracle tests in the scheduler notebook do the same.

In [ ]:
import math
from struct import pack, unpack
from collections import namedtuple
from fastcore.utils import *

In [ ]:
from fastcore.test import *

## Parameters

In [ ]:
S_MIN,S_MAX,D_MIN,D_MAX = 0.001,36500.0,1.0,10.0
FSRS5_DECAY,FSRS6_DECAY = 0.5,0.1542

DEFAULT_PARAMS = [0.212,1.2931,2.3065,8.2956,6.4133,0.8334,3.0194,0.001,1.8722,0.1666,0.796,
    1.4835,0.0614,0.2629,1.6483,0.6014,1.8729,0.5425,0.0912,0.0658,FSRS6_DECAY]

_CLAMPS = [(S_MIN,100.)]*4 + [(D_MIN,D_MAX),(0.001,4.0),(0.001,4.0),(0.001,0.75),(0.,4.5),(0.,0.8),(0.001,3.5),
    (0.001,5.0),(0.001,0.25),(0.001,0.9),(0.,4.0),(0.,1.0),(1.0,6.0),(0.,2.0),(0.,2.0),(0.,0.8),(0.1,0.8)]

def f32(x): return unpack('f', pack('f', x))[0]
def clamp(x, lo, hi): return min(max(x,lo),hi)

def fsrs_params(w):
    "Upgrade a 0/17/19/21-length parameter list to FSRS-6's 21 numbers and clip to legal ranges, like `FSRS::new`"
    w = [float(x) for x in w] or list(DEFAULT_PARAMS)
    if len(w)==17:
        w[4],w[5],w[6] = w[4]+2*w[5], math.log(w[5]*3+1)/3, w[6]+0.5
        w += [0.,0.,0.,FSRS5_DECAY]
    elif len(w)==19: w += [0.,FSRS5_DECAY]
    assert len(w)==21, f"invalid FSRS parameter count: {len(w)}"
    return [f32(clamp(x,*b)) for x,b in zip(w,_CLAMPS)]

def param_decay(w):
    "The forgetting-curve decay for a *raw* (pre-upgrade) parameter list, Anki's `get_decay_from_params`"
    return FSRS6_DECAY if not len(w) else (FSRS5_DECAY if len(w)<21 else w[20])

In [ ]:
test_eq(len(fsrs_params([])), 21)
test_eq(fsrs_params([]), [f32(x) for x in DEFAULT_PARAMS])
up = fsrs_params([0.4,0.6,2.4,5.8,4.93,0.94,0.86,0.01,1.49,0.14,0.94,2.18,0.05,0.34,1.26,0.29,2.61])
test_eq(len(up), 21)
test_close(up[4], 4.93+2*0.94, eps=1e-4)  # FSRS-4.5's difficulty params shift on upgrade
test_close(up[20], FSRS5_DECAY, eps=1e-6)  # ...and keep the old fixed decay
test_eq(fsrs_params([0.1]*19)[18:], [f32(0.1), 0.0, FSRS5_DECAY])  # 19-param sets gain w19=0 and the fixed FSRS-5 decay
test_eq(param_decay([]), FSRS6_DECAY)
test_eq(param_decay([0.1]*17), FSRS5_DECAY)
test_eq(param_decay(DEFAULT_PARAMS), FSRS6_DECAY)

## The forgetting curve

By construction, retrievability is exactly 0.9 when the elapsed time equals the stability, whatever the decay — and the next interval inverts the curve, so at desired retention 0.9, the interval *is* the stability:

In [ ]:
def forgetting_curve(w, t, s):
    "Probability of recall `t` days after a review that left stability `s`"
    decay = -w[20]
    factor = math.exp(math.log(0.9)/decay) - 1
    return f32((t/s*factor + 1)**decay)

def next_interval(w, s, dr):
    "The (fractional) days until retrievability falls to desired retention `dr`, at stability `s`"
    decay = -w[20]
    factor = math.exp(math.log(0.9)/decay) - 1
    return f32(s/factor*(dr**(1/decay) - 1))

In [ ]:
w = fsrs_params([])
test_close(forgetting_curve(w, 5, 5), 0.9, eps=1e-6)
test_close(next_interval(w, 5, 0.9), 5, eps=1e-5)
assert next_interval(w, 5, 0.95) < 5 < next_interval(w, 5, 0.8)  # higher retention -> shorter intervals
forgetting_curve(w, 30, 5), forgetting_curve(w, 1, 5)

(0.742717444896698, 0.972769558429718)

## Memory state updates

`step` is one review: a rating (1=Again, ..., 4=Easy) after `delta_t` days. The first rating of a new card reads its initial state straight from the first four parameters (stability) and `w4`/`w5` (difficulty). After that:
- difficulty moves by the rating with linear damping and reversion toward the Easy-init mean
- a pass multiplies stability by a factor that grows with ease of recall (and shrinks under `w15` for Hard, grows under `w16` for Easy)
- a lapse rebuilds stability from scratch, capped so it can't exceed the pre-lapse value
- and a *same-day* review (`delta_t` 0) uses the separate short-term formula in `w17`-`w19`.

In [ ]:
class MemSt(namedtuple('MemSt', 'stability difficulty')):
    "An FSRS memory state: `stability` in days, `difficulty` in 1-10"

def _init_d(w, r): return w[4] - math.exp(w[5]*(r-1)) + 1

def _next_d(w, d, r):
    nd = d + (10-d)*(-w[6]*(r-3))/9
    return clamp(w[7]*(_init_d(w,4)-nd)+nd, D_MIN, D_MAX)

def _s_success(w, s, d, r, rating):
    hp = w[15] if rating==2 else 1.0
    eb = w[16] if rating==4 else 1.0
    return s*(math.exp(w[8])*(11-d)*s**-w[9]*(math.exp((1-r)*w[10])-1)*hp*eb + 1)

def _s_fail(w, s, d, r):
    ns = w[11]*d**-w[12]*((s+1)**w[13]-1)*math.exp((1-r)*w[14])
    return min(ns, s/math.exp(w[17]*w[18]))

def _s_short_term(w, s, rating):
    "Same-day stability. Floors the multiplier at 1 for Good/Easy only"
    sinc = math.exp(w[17]*(rating-3+w[18]))*s**-w[19]
    return s*(max(sinc, 1.0) if rating>=3 else sinc)

def step(w, delta_t, rating, mem):
    "Memory state after rating a card `delta_t` days since its last review (`mem` None: first rating of a new card)"
    if mem is None:
        r = clamp(rating, 1, 4)
        return MemSt(f32(clamp(w[r-1], S_MIN, S_MAX)), f32(clamp(_init_d(w,r), D_MIN, D_MAX)))
    s,d = clamp(mem.stability, S_MIN, S_MAX), clamp(mem.difficulty, D_MIN, D_MAX)
    r = forgetting_curve(w, delta_t, s)
    if delta_t==0:  ns = _s_short_term(w, s, rating)
    elif rating==1: ns = _s_fail(w, s, d, r)
    else:           ns = _s_success(w, s, d, r, rating)
    return MemSt(f32(clamp(ns, S_MIN, S_MAX)), f32(_next_d(w, d, rating)))

ItemSt = namedtuple('ItemSt', 'mem ivl')

def next_states(w, mem, dr, days_elapsed):
    "For each rating 1-4 (index `[ease-1]`), the next `MemSt` and its desired-retention interval"
    sts = [step(w, float(days_elapsed), rating, mem) for rating in (1,2,3,4)]
    return [ItemSt(m, next_interval(w, m.stability, dr)) for m in sts]

The check below is fsrs-rs's own doc-test: a new card at desired retention 0.9 lands on these exact states for each button.

In [ ]:
ns = next_states(w, None, 0.9, 0)
for got,(ws,wd) in zip(ns, [(0.212,6.4133),(1.2931,5.1121707),(2.3065,2.118104),(8.2956,1.0)]):
    test_close((got.mem.stability, got.mem.difficulty, got.ivl), (ws, wd, ws), eps=1e-4)
ns[2]

ItemSt(mem=MemSt(stability=2.30649995803833, difficulty=2.1181039810180664), ivl=2.30649995803833)

## Approximating memory state from SM-2

A card that has scheduling state but no usable review history (imported with truncated revlogs, say) gets a memory state approximated from its SM-2 ease and interval, assuming the user's historical retention. The scheduler uses this as the starting state when replaying an incomplete revlog.

In [ ]:
def memory_state_from_sm2(w, ease_factor, interval, sm2_retention=0.9):
    "A `MemSt` inferred from SM-2 `ease_factor` (eg 2.5) and `interval` days"
    decay = -w[20]
    factor = 0.9**(1/decay) - 1
    s = max(interval, S_MIN)*factor/(sm2_retention**(1/decay) - 1)
    d = 11 - (ease_factor-1)/(math.exp(w[8])*s**-w[9]*(math.exp((1-sm2_retention)*w[10]) - 1))
    return MemSt(f32(s), f32(clamp(d, D_MIN, D_MAX)))

In [ ]:
m = memory_state_from_sm2(w, 2.5, 10)
test_close(m.stability, 10, eps=1e-4)  # at the assumed retention, stability ~= the SM-2 interval
assert 1 <= m.difficulty <= 10
m2 = memory_state_from_sm2(w, 1.3, 10)
assert m2.difficulty > m.difficulty  # low ease reads as high difficulty
m

MemSt(stability=10.0, difficulty=6.914055347442627)